In [1]:
# YOLOv5 Object Counter with Line Crossing Detection
# By: GitHub Copilot for mmaleki92 (2025-02-27)

import os
import sys
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output
from google.colab import files
from pathlib import Path
import time
import urllib.request
from collections import defaultdict

# Install YOLOv5 if not already installed
def install_yolov5():
    if not os.path.exists('yolov5'):
        print("Installing YOLOv5...")
        # Clone YOLOv5 repository
        !git clone https://github.com/ultralytics/yolov5
        # Install YOLOv5 requirements
        !pip install -q -r yolov5/requirements.txt
        # Add yolov5 directory to path
        sys.path.append('./yolov5')
        print("YOLOv5 installed successfully!")
    else:
        print("YOLOv5 is already installed.")
        sys.path.append('./yolov5')

# Function to load YOLOv5 model
def load_model(model_size='s'):
    # Valid model sizes: n, s, m, l, x
    valid_sizes = ['n', 's', 'm', 'l', 'x']
    if model_size not in valid_sizes:
        print(f"Invalid model size. Using 's' instead. Valid sizes are: {', '.join(valid_sizes)}")
        model_size = 's'

    model = torch.hub.load('ultralytics/yolov5', f'yolov5{model_size}', pretrained=True)
    return model

# Function to count objects in an image
def count_objects(results):
    # Get detected objects
    df = results.pandas().xyxy[0]

    # Count objects by class
    object_counts = {}
    for class_name in df['name'].unique():
        count = len(df[df['name'] == class_name])
        object_counts[class_name] = count

    return object_counts, df

# Function to draw bounding boxes and counts on image
def draw_boxes_and_counts(img, df, counts, line_y=None, crossed_counts=None):
    # Make a copy of the image to avoid modifying the original
    img_with_boxes = img.copy()

    # Draw the counting line if specified
    if line_y is not None:
        cv2.line(img_with_boxes, (0, line_y), (img.shape[1], line_y), (0, 255, 255), 2)
        # Add a direction arrow
        arrow_length = 30
        arrow_x = img.shape[1] // 2
        cv2.arrowedLine(img_with_boxes, (arrow_x, line_y + arrow_length),
                       (arrow_x, line_y - arrow_length), (0, 255, 255), 2, tipLength=0.3)

    # Draw each bounding box
    for idx, row in df.iterrows():
        xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
        class_name = row['name']
        confidence = row['confidence']

        # Generate a consistent color for this class
        color = (int(hash(class_name) % 255),
                 int(hash(class_name + '1') % 255),
                 int(hash(class_name + '2') % 255))

        # Draw rectangle and label
        cv2.rectangle(img_with_boxes, (xmin, ymin), (xmax, ymax), color, 2)
        label = f"{class_name}: {confidence:.2f}"
        cv2.putText(img_with_boxes, label, (xmin, ymin - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Add regular detection count information at the top left
    y_pos = 30
    for class_name, count in counts.items():
        text = f"Detected {class_name}: {count}"
        cv2.putText(img_with_boxes, text, (10, y_pos),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        y_pos += 30

    # Add crossing counts at the top right if available
    if crossed_counts:
        y_pos = 30
        x_pos = img.shape[1] - 300
        cv2.putText(img_with_boxes, "OBJECTS CROSSING LINE (↑):",
                   (x_pos, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        y_pos += 30
        for class_name, count in crossed_counts.items():
            text = f"{class_name}: {count}"
            cv2.putText(img_with_boxes, text, (x_pos, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            y_pos += 30

    return img_with_boxes

# Process image file
def process_image(model, image_path, conf_threshold=0.25):
    # Load image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error loading image from {image_path}")
        return None, None

    # Convert BGR to RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Set confidence threshold
    model.conf = conf_threshold

    # Perform inference
    results = model(img_rgb)

    # Count objects
    counts, detections = count_objects(results)

    # Draw boxes and counts
    img_with_boxes = draw_boxes_and_counts(img_rgb, detections, counts)

    return img_with_boxes, counts

# Process video file with line crossing detection
def process_video(model, video_path, output_path=None, conf_threshold=0.25):
    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return None, None

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Define the counting line at the middle height of the video
    line_y = height // 2

    # Create output video writer if output path is specified
    writer = None
    if output_path:
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Set confidence threshold
    model.conf = conf_threshold

    # Dictionary to track object positions
    object_tracks = {}
    # Dictionary to count objects that cross the line (from bottom to top)
    crossed_counts = defaultdict(int)
    # Set to keep track of objects that have already crossed the line
    already_counted = set()

    frame_count = 0
    start_time = time.time()
    all_counts = {}

    # Process each frame
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Perform inference with tracking
        results = model.track(frame_rgb, persist=True)

        # Get detection and tracking information
        if hasattr(results, 'pandas') and hasattr(results.pandas(), 'xyxy'):
            df = results.pandas().xyxy[0]

            # Process each detection
            for idx, row in df.iterrows():
                # Skip if no tracking ID
                if 'id' not in row:
                    continue

                obj_id = row['id']
                class_name = row['name']
                xmin, ymin, xmax, ymax = row['xmin'], row['ymin'], row['xmax'], row['ymax']

                # Calculate the bottom center of the bounding box
                center_x = (xmin + xmax) / 2
                center_y = ymax  # Use the bottom of the bounding box

                # Track this object's position
                if obj_id not in object_tracks:
                    object_tracks[obj_id] = {'positions': [], 'class': class_name}
                object_tracks[obj_id]['positions'].append((center_x, center_y))

                # Keep only the last 2 positions to detect line crossing
                if len(object_tracks[obj_id]['positions']) > 2:
                    object_tracks[obj_id]['positions'] = object_tracks[obj_id]['positions'][-2:]

                # Check if object has crossed the line from bottom to top
                if len(object_tracks[obj_id]['positions']) == 2 and obj_id not in already_counted:
                    prev_y = object_tracks[obj_id]['positions'][0][1]
                    curr_y = object_tracks[obj_id]['positions'][1][1]

                    # Check if the object has crossed the line from below to above
                    if prev_y > line_y and curr_y <= line_y:
                        crossed_counts[class_name] += 1
                        already_counted.add(obj_id)

        # Count objects in the current frame
        counts, detections = count_objects(results)

        # Update total counts
        for class_name, count in counts.items():
            if class_name in all_counts:
                all_counts[class_name] = max(all_counts[class_name], count)
            else:
                all_counts[class_name] = count

        # Draw boxes, the counting line, and counts
        frame_with_boxes = draw_boxes_and_counts(frame_rgb, detections, counts, line_y, crossed_counts)

        # Convert RGB back to BGR for video writing
        frame_with_boxes_bgr = cv2.cvtColor(frame_with_boxes, cv2.COLOR_RGB2BGR)

        if writer:
            writer.write(frame_with_boxes_bgr)

        # Display progress
        frame_count += 1
        if frame_count % 30 == 0:  # Update every 30 frames
            elapsed_time = time.time() - start_time
            fps_processed = frame_count / elapsed_time
            print(f"Processed {frame_count} frames. FPS: {fps_processed:.2f}", end="\r")

    # Release resources
    cap.release()
    if writer:
        writer.release()

    print(f"\nVideo processing complete. Processed {frame_count} frames.")
    return all_counts, crossed_counts

# Function to download an example image or video
def download_example_file(url, filename):
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filename)
    print(f"Downloaded to {filename}")
    return filename

# Main function
def main():
    # Install YOLOv5
    install_yolov5()

    # Load model
    print("Loading YOLOv5 model...")
    model = load_model('s')  # Load the small model
    print("Model loaded successfully!")

    # Ask user for input type
    print("\nWhat would you like to process?")
    print("1. Example image (street scene)")
    print("2. Example video (traffic)")
    print("3. Upload your own image")
    print("4. Upload your own video")
    print("5. Use a YouTube video")

    choice = input("Enter your choice (1-5): ")

    if choice == '1':
        # Download example image
        image_url = "https://ultralytics.com/images/zidane.jpg"
        image_path = download_example_file(image_url, "example_image.jpg")

        # Process image
        result_img, counts = process_image(model, image_path)

        # Display results
        plt.figure(figsize=(12, 8))
        plt.imshow(result_img)
        plt.title("Object Detection Results")
        plt.axis('off')
        plt.show()

        print("\nObject counts:")
        for class_name, count in counts.items():
            print(f"{class_name}: {count}")

    elif choice == '2':
        # Download example video
        video_url = "https://ultralytics.com/assets/highway.mp4"
        video_path = download_example_file(video_url, "example_video.mp4")
        output_path = "result_video.mp4"

        # Process video
        all_counts, crossed_counts = process_video(model, video_path, output_path)

        print("\nMaximum object counts throughout the video:")
        for class_name, count in all_counts.items():
            print(f"{class_name}: {count}")

        print("\nObjects that crossed the line (bottom to top):")
        for class_name, count in crossed_counts.items():
            print(f"{class_name}: {count}")

        print(f"\nProcessed video saved as {output_path}")
        files.download(output_path)

    elif choice == '3':
        # Upload image
        print("Please upload an image...")
        uploaded = files.upload()

        if uploaded:
            image_path = next(iter(uploaded.keys()))

            # Process image
            result_img, counts = process_image(model, image_path)

            # Display results
            plt.figure(figsize=(12, 8))
            plt.imshow(result_img)
            plt.title("Object Detection Results")
            plt.axis('off')
            plt.show()

            print("\nObject counts:")
            for class_name, count in counts.items():
                print(f"{class_name}: {count}")

            # Save and provide the result
            result_path = "detection_result.jpg"
            cv2.imwrite(result_path, cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
            print(f"\nResult saved as {result_path}")
            files.download(result_path)

    elif choice == '4':
        # Upload video
        print("Please upload a video...")
        uploaded = files.upload()

        if uploaded:
            video_path = next(iter(uploaded.keys()))
            output_path = "result_video.mp4"

            # Process video
            all_counts, crossed_counts = process_video(model, video_path, output_path)

            print("\nMaximum object counts throughout the video:")
            for class_name, count in all_counts.items():
                print(f"{class_name}: {count}")

            print("\nObjects that crossed the line (bottom to top):")
            for class_name, count in crossed_counts.items():
                print(f"{class_name}: {count}")

            print(f"\nProcessed video saved as {output_path}")
            files.download(output_path)

    elif choice == '5':
        # For YouTube videos, we need pytube
        try:
            from pytube import YouTube
        except ImportError:
            print("Installing pytube for YouTube video downloads...")
            !pip install pytube
            from pytube import YouTube

        # Get YouTube URL
        youtube_url = input("Enter YouTube URL: ")

        try:
            # Create a YouTube object
            yt = YouTube(youtube_url)
            print(f"Downloading: {yt.title}")

            # Get the video with progressive stream (with audio) and mp4
            stream = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').desc().first()

            # Download the video
            video_path = stream.download(filename="youtube_video.mp4")
            print(f"Download complete: {video_path}")

            output_path = "result_youtube_video.mp4"

            # Process video
            all_counts, crossed_counts = process_video(model, video_path, output_path)

            print("\nMaximum object counts throughout the video:")
            for class_name, count in all_counts.items():
                print(f"{class_name}: {count}")

            print("\nObjects that crossed the line (bottom to top):")
            for class_name, count in crossed_counts.items():
                print(f"{class_name}: {count}")

            print(f"\nProcessed video saved as {output_path}")
            files.download(output_path)

        except Exception as e:
            print(f"Error processing YouTube video: {e}")

    else:
        print("Invalid choice. Please run again and select a valid option.")

if __name__ == "__main__":
    main()

Installing YOLOv5...
Cloning into 'yolov5'...
remote: Enumerating objects: 17270, done.
remote: Counting objects: 100% (1/1), done.
remote: Total 17270 (delta 0), reused 0 (delta 0), pack-reused 17269 (from 2)
Receiving objects: 100% (17270/17270), 16.11 MiB | 18.87 MiB/s, done.
Resolving deltas: 100% (11861/11861), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.11/dist-packages/torch/hub.py:330: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


YOLOv5 🚀 2025-2-27 Python-3.11.11 torch-2.5.1+cu124 CPU

100%|██████████| 14.1M/14.1M [00:00<00:00, 134MB/s]

Fusing layers... 
YOLOv5s summary: 213 layers, 7225885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 


Model loaded successfully!

What would you like to process?
1. Example image (street scene)
2. Example video (traffic)
3. Upload your own image
4. Upload your own video
5. Use a YouTube video
Enter your choice (1-5): 4
Please upload a video...


Saving background video _ people _ walking _(1).mp4 to background video _ people _ walking _(1).mp4


AttributeError: 'AutoShape' object has no attribute 'track'

In [4]:
# YOLOv8 Object Counter with Line Crossing Detection
# By: GitHub Copilot for mmaleki92 (2025-02-27)

import os
import sys
import cv2
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image, clear_output
from google.colab import files
from pathlib import Path
import time
import urllib.request
from collections import defaultdict

# Install YOLOv8 if not already installed
def install_requirements():
    print("Installing Ultralytics YOLOv8...")
    !pip install -q ultralytics
    print("YOLOv8 installed successfully!")

    # For YouTube videos
    print("Installing pytube for YouTube downloads...")
    !pip install -q pytube
    print("Pytube installed successfully!")

# Function to load YOLOv8 model
def load_model(model_size='s'):
    from ultralytics import YOLO

    # Valid model sizes: n, s, m, l, x
    valid_sizes = ['n', 's', 'm', 'l', 'x']
    if model_size not in valid_sizes:
        print(f"Invalid model size. Using 's' instead. Valid sizes are: {', '.join(valid_sizes)}")
        model_size = 's'

    print(f"Loading YOLOv8{model_size} model...")
    model = YOLO(f"yolov8{model_size}.pt")
    print("Model loaded successfully!")
    return model

# Function to count objects in an image
def count_objects(results):
    object_counts = defaultdict(int)

    # Get all detected objects
    if not results or not hasattr(results[0], 'boxes'):
        return object_counts, []

    # Get classes and class names
    class_ids = results[0].boxes.cls.cpu().numpy()
    for class_id in class_ids:
        class_name = results[0].names[int(class_id)]
        object_counts[class_name] += 1

    return object_counts, results[0].boxes

# Function to draw bounding boxes and counts on image
def draw_boxes_and_counts(img, results, counts, line_y=None, crossed_counts=None):
    # Make a copy of the image to avoid modifying the original
    img_with_boxes = img.copy()

    # Draw the counting line if specified
    if line_y is not None:
        cv2.line(img_with_boxes, (0, line_y), (img.shape[1], line_y), (0, 255, 255), 2)
        # Add a direction arrow
        arrow_length = 30
        arrow_x = img.shape[1] // 2
        cv2.arrowedLine(img_with_boxes, (arrow_x, line_y + arrow_length),
                       (arrow_x, line_y - arrow_length), (0, 255, 255), 2, tipLength=0.3)
        # Add a label for the line
        cv2.putText(img_with_boxes, "Counting Line", (arrow_x + 10, line_y - 10),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)

    # Add regular detection count information at the top left
    y_pos = 30
    for class_name, count in counts.items():
        text = f"Detected {class_name}: {count}"
        cv2.putText(img_with_boxes, text, (10, y_pos),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        y_pos += 30

    # Add crossing counts at the top right if available
    if crossed_counts:
        y_pos = 30
        x_pos = img.shape[1] - 300
        cv2.putText(img_with_boxes, "OBJECTS CROSSING LINE (↑):",
                   (x_pos, y_pos), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
        y_pos += 30
        for class_name, count in crossed_counts.items():
            text = f"{class_name}: {count}"
            cv2.putText(img_with_boxes, text, (x_pos, y_pos),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
            y_pos += 30

    # Use the built-in plotting from YOLOv8 if results are available
    if results and hasattr(results[0], 'plot'):
        plotted_img = results[0].plot()
        # Combine the plotted image with our counting information
        # by taking the boxes from plotted_img and keeping our text
        mask = np.all(plotted_img == img, axis=-1)
        img_with_boxes[~mask] = plotted_img[~mask]

    return img_with_boxes

# Process image file
def process_image(model, image_path, conf_threshold=0.25):
    # Set confidence threshold
    model.conf = conf_threshold

    # Perform inference
    results = model(image_path)

    # Load image for drawing
    img = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Count objects
    counts, _ = count_objects(results)

    # Draw boxes and counts
    img_with_boxes = draw_boxes_and_counts(img_rgb, results, counts)

    return img_with_boxes, counts

# Process video file with line crossing detection
def process_video(model, video_path, output_path=None, conf_threshold=0.25):
    # Open video file
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f"Error opening video file {video_path}")
        return None, None

    # Get video properties
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    # Define the counting line at the middle height of the video
    line_y = height // 2

    # Create output video writer if output path is specified
    writer = None
    if output_path:
        writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Set confidence threshold
    model.conf = conf_threshold

    # Dictionary to track object positions
    object_tracks = {}
    # Dictionary to count objects that cross the line (from bottom to top)
    crossed_counts = defaultdict(int)
    # Set to keep track of objects that have already crossed the line
    already_counted = set()

    frame_count = 0
    start_time = time.time()
    all_counts = defaultdict(int)

    print("Processing video frames...")

    # Process each frame
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Convert BGR to RGB
        frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # Perform inference with tracking enabled
        results = model.track(frame_rgb, persist=True)

        # Count objects in the current frame
        counts, boxes = count_objects(results)

        # Update total counts
        for class_name, count in counts.items():
            all_counts[class_name] = max(all_counts[class_name], count)

        # Process tracking information if available
        if hasattr(results[0], 'boxes') and hasattr(results[0].boxes, 'id') and results[0].boxes.id is not None:
            track_ids = results[0].boxes.id.int().cpu().numpy()
            boxes_xyxy = results[0].boxes.xyxy.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy()

            for box, track_id, class_id in zip(boxes_xyxy, track_ids, classes):
                x1, y1, x2, y2 = box
                class_name = results[0].names[int(class_id)]

                # Calculate the bottom center of the bounding box
                center_x = (x1 + x2) / 2
                bottom_y = y2

                # Track this object's position
                if track_id not in object_tracks:
                    object_tracks[track_id] = {'positions': [], 'class': class_name}
                object_tracks[track_id]['positions'].append((center_x, bottom_y))

                # Keep only the last 2 positions to detect line crossing
                if len(object_tracks[track_id]['positions']) > 2:
                    object_tracks[track_id]['positions'] = object_tracks[track_id]['positions'][-2:]

                # Check if object has crossed the line from bottom to top
                if len(object_tracks[track_id]['positions']) >= 2 and track_id not in already_counted:
                    prev_y = object_tracks[track_id]['positions'][-2][1]
                    curr_y = object_tracks[track_id]['positions'][-1][1]

                    # Check if the object has crossed the line from below to above
                    if prev_y > line_y and curr_y <= line_y:
                        crossed_counts[class_name] += 1
                        already_counted.add(track_id)

        # Draw boxes, the counting line, and counts
        frame_with_boxes = draw_boxes_and_counts(frame_rgb, results, counts, line_y, crossed_counts)

        # Convert RGB back to BGR for video writing
        frame_with_boxes_bgr = cv2.cvtColor(frame_with_boxes, cv2.COLOR_RGB2BGR)

        if writer:
            writer.write(frame_with_boxes_bgr)

        # Display progress
        frame_count += 1
        if frame_count % 30 == 0:  # Update every 30 frames
            elapsed_time = time.time() - start_time
            fps_processed = frame_count / elapsed_time
            percent_complete = (frame_count / total_frames) * 100 if total_frames > 0 else 0
            print(f"Processed {frame_count}/{total_frames} frames ({percent_complete:.1f}%). FPS: {fps_processed:.2f}", end="\r")

    # Release resources
    cap.release()
    if writer:
        writer.release()

    print(f"\nVideo processing complete. Processed {frame_count} frames.")
    return all_counts, crossed_counts

# Function to download an example image or video
def download_example_file(url, filename):
    print(f"Downloading {filename}...")
    urllib.request.urlretrieve(url, filename)
    print(f"Downloaded to {filename}")
    return filename

# Function to download YouTube video
def download_youtube_video(url, output_path="youtube_video.mp4"):
    try:
        from pytube import YouTube
    except ImportError:
        print("Installing pytube for YouTube video downloads...")
        !pip install -q pytube
        from pytube import YouTube

    try:
        # Create a YouTube object
        yt = YouTube(url)
        print(f"Downloading: {yt.title}")

        # Get the video with progressive stream (with audio) and mp4
        stream = yt.streams.filter(progressive=True, file_extension='mp4').order_by('resolution').desc().first()

        # Download the video
        stream.download(filename=output_path)
        print(f"Download complete: {output_path}")

        return output_path, yt.title
    except Exception as e:
        print(f"Error downloading YouTube video: {e}")
        return None, None

# Main function
def main():
    # Install requirements
    install_requirements()

    # Load model
    model = load_model('n')  # Load the nano model for faster processing

    # Ask user for input type
    print("\nWhat would you like to process?")
    print("1. Example image (street scene)")
    print("2. Example video (traffic)")
    print("3. Upload your own image")
    print("4. Upload your own video")
    print("5. Use a YouTube video")

    choice = input("Enter your choice (1-5): ")

    if choice == '1':
        # Download example image
        image_url = "https://ultralytics.com/images/zidane.jpg"
        image_path = download_example_file(image_url, "example_image.jpg")

        # Process image
        result_img, counts = process_image(model, image_path)

        # Display results
        plt.figure(figsize=(12, 8))
        plt.imshow(result_img)
        plt.title("Object Detection Results")
        plt.axis('off')
        plt.show()

        print("\nObject counts:")
        for class_name, count in counts.items():
            print(f"{class_name}: {count}")

    elif choice == '2':
        # Download example video
        video_url = "https://ultralytics.com/assets/highway.mp4"
        video_path = download_example_file(video_url, "example_video.mp4")
        output_path = "result_video.mp4"

        # Process video
        all_counts, crossed_counts = process_video(model, video_path, output_path)

        print("\nMaximum object counts throughout the video:")
        for class_name, count in all_counts.items():
            print(f"{class_name}: {count}")

        print("\nObjects that crossed the line (bottom to top):")
        for class_name, count in crossed_counts.items():
            print(f"{class_name}: {count}")

        print(f"\nProcessed video saved as {output_path}")
        files.download(output_path)

    elif choice == '3':
        # Upload image
        print("Please upload an image...")
        uploaded = files.upload()

        if uploaded:
            image_path = next(iter(uploaded.keys()))

            # Process image
            result_img, counts = process_image(model, image_path)

            # Display results
            plt.figure(figsize=(12, 8))
            plt.imshow(result_img)
            plt.title("Object Detection Results")
            plt.axis('off')
            plt.show()

            print("\nObject counts:")
            for class_name, count in counts.items():
                print(f"{class_name}: {count}")

            # Save and provide the result
            result_path = "detection_result.jpg"
            cv2.imwrite(result_path, cv2.cvtColor(result_img, cv2.COLOR_RGB2BGR))
            print(f"\nResult saved as {result_path}")
            files.download(result_path)

    elif choice == '4':
        # Upload video
        print("Please upload a video...")
        uploaded = files.upload()

        if uploaded:
            video_path = next(iter(uploaded.keys()))
            output_path = "result_video.mp4"

            # Process video
            all_counts, crossed_counts = process_video(model, video_path, output_path)

            print("\nMaximum object counts throughout the video:")
            for class_name, count in all_counts.items():
                print(f"{class_name}: {count}")

            print("\nObjects that crossed the line (bottom to top):")
            for class_name, count in crossed_counts.items():
                print(f"{class_name}: {count}")

            print(f"\nProcessed video saved as {output_path}")
            files.download(output_path)

    elif choice == '5':
        # Get YouTube URL
        youtube_url = input("Enter YouTube URL: ")

        # Download YouTube video
        video_path, video_title = download_youtube_video(youtube_url)

        if video_path:
            output_path = "result_youtube_video.mp4"

            # Process video
            all_counts, crossed_counts = process_video(model, video_path, output_path)

            print("\nMaximum object counts throughout the video:")
            for class_name, count in all_counts.items():
                print(f"{class_name}: {count}")

            print("\nObjects that crossed the line (bottom to top):")
            for class_name, count in crossed_counts.items():
                print(f"{class_name}: {count}")

            print(f"\nProcessed video saved as {output_path}")
            files.download(output_path)

    else:
        print("Invalid choice. Please run again and select a valid option.")

# Example code for direct execution with a specific YouTube video
def process_example_youtube():
    # Install requirements
    install_requirements()

    # Load model
    model = load_model('n')  # Load the nano model for faster processing

    # Specific traffic video from YouTube
    youtube_url = "https://www.youtube.com/watch?v=ORrrKXGx2SE"  # Traffic junction video

    # Download YouTube video
    video_path, video_title = download_youtube_video(youtube_url)

    if video_path:
        output_path = "result_youtube_video.mp4"

        # Process video
        all_counts, crossed_counts = process_video(model, video_path, output_path)

        print("\nResults for video:", video_title)
        print("\nMaximum object counts throughout the video:")
        for class_name, count in all_counts.items():
            print(f"{class_name}: {count}")

        print("\nObjects that crossed the line (bottom to top):")
        for class_name, count in crossed_counts.items():
            print(f"{class_name}: {count}")

        print(f"\nProcessed video saved as {output_path}")
        files.download(output_path)

if __name__ == "__main__":
    main()
    # Uncomment below to directly process an example YouTube video instead of showing the menu
    # process_example_youtube()

Installing Ultralytics YOLOv8...
YOLOv8 installed successfully!
Installing pytube for YouTube downloads...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.8 MB/s eta 0:00:00
Pytube installed successfully!
Loading YOLOv8n model...


100%|██████████| 6.25M/6.25M [00:00<00:00, 71.1MB/s]


Model loaded successfully!

What would you like to process?
1. Example image (street scene)
2. Example video (traffic)
3. Upload your own image
4. Upload your own video
5. Use a YouTube video
Enter your choice (1-5): 4
Please upload a video...


Saving background video _ people _ walking _(1).mp4 to background video _ people _ walking _(1) (1).mp4
Processing video frames...
requirements: Ultralytics requirement ['lap>=0.5.12'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.3 MB/s eta 0:00:00

requirements: AutoUpdate success ✅ 2.8s, installed 1 package: ['lap>=0.5.12']
requirements: ⚠️ Restart runtime or rerun command for updates to take effect


0: 384x640 36 persons, 2 birds, 177.8ms
Speed: 18.7ms preprocess, 177.8ms inference, 51.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 34 persons, 2 birds, 155.2ms
Speed: 2.7ms preprocess, 155.2ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 2 birds, 171.7ms
Speed: 3.7ms preprocess, 171.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 35 persons, 2 birds, 156.0ms
Speed: 4.0ms preprocess, 156.0ms inference, 1.6ms postprocess per image at shape (1, 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>